# Train orientation_strong + scale_strong seeds 0–11 + probe (Kaggle)

Trains all **24** encoders — 12 `orientation_strong` (Shapes3D, rotation-only,
±180°) and 12 `scale_strong` (zoom-only, 0.6–1.4×) — from scratch to the
50-epoch cut, then probes each finished cell into its `stacks.npz`. Both are
Shapes3D cells, so they share one image cache and draw from a single unified
GPU queue: with `GPU T4 x2` the two conditions run side by side (one seed per
GPU), i.e. both train at once.

Everything is checkpointed and resumable: encoders per epoch (`last_ckpt.pt`)
and per finished seed (`backbone.pt`), probes per finished encoder
(`_cache/*.npz`). Stop after any session and re-run from the top to continue —
sections 5–6 restore every checkpoint before training resumes.

## One-time setup
1. **Settings → Accelerator → `GPU T4 x2`** (recommended — runs both conditions
   at once; a single T4 also works, just serial).
2. **Settings → Internet → On** (clone + install + Shapes3D download).

## Run order
Sections 1–6 top to bottom (clone, install, data, restore), then **section 7**
(train). Section 7 is idempotent + resumable: it skips a seed once its
`backbone.pt` exists and resumes a partial seed from its own `last_ckpt.pt`, so
**re-run it after a session ends** to continue. **Save Version** every few hours
so `/kaggle/working` persists (Kaggle sessions cap ~12 h), then
`Add Input -> this notebook's previous version` next session so section 5 can
restore the checkpoints.

**New-condition check:** these are the first cells trained with the orientation
and scale augmentations, so once the first seeds of each land run section 9 and
check the collapse diagnostics — verify the 50-epoch cut is healthy before
letting the rest of the quota burn.

When all 12 encoders of a condition exist, **section 10** probes it into
`results/probes/<condition>_strong/`. It probes both cells in parallel (one per
GPU), is resumable per-encoder (`--resume`), and skips a cell whose `stacks.npz`
already exists. The per-encoder quality gate runs inside the probe step;
gate-failed seeds are recorded in `meta.json` and excluded downstream.

**Wall-clock (2× T4):** ~5 GPU-h per Shapes3D encoder; 24 encoders ÷ 2 GPUs ≈
**~60 h** training ≈ ~6 sessions (~2 weeks at 30 quota-h/wk), plus **~3–4 h**
for the parallel probe step. See the estimate note at the bottom.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")

## 4. Download Shapes3D + build the image cache
Both cells are Shapes3D, so this runs once and both conditions share it.
`--build-cache` decompresses into an uncompressed memmap the trainers mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 5. Restore checkpoints from a previous session (resume)
`Add Input -> this notebook's previous version's output` (or your uploaded encoder
dataset), then run this. It restores, for **both** conditions:
- encoder checkpoints — every `*<condition>_strong_seed<N>*.pt` under
  `/kaggle/input` -> `results/encoders/<condition>_strong_seed<N>/`
  (`backbone*.pt` -> `backbone.pt`, `last_ckpt*.pt` -> `last_ckpt.pt`);
- probe artifacts — `results/probes/<condition>_strong/{stacks.npz, meta.json,
  _cache/*.npz}` so the probe step resumes per-encoder.

On a fresh first run there is nothing to restore -- it says so and you train from
scratch. Only orientation/scale files are touched; existing local files are kept.

In [ ]:
import re, shutil
from pathlib import Path

REPO  = Path("/kaggle/working/probe-capacity-invariance")
ENC   = REPO / "results" / "encoders"
PROB  = REPO / "results" / "probes"
INPUT = Path("/kaggle/input")
_RID  = re.compile(r"(?:orientation|scale)_strong_seed\d+")
_PROBE = re.compile(r"results/probes/((?:orientation|scale)_strong/.+)$")

def _target(name):
    n = name.lower()
    if "ckpt" in n:     return "last_ckpt.pt"
    if "backbone" in n: return "backbone.pt"
    return None

# --- encoder checkpoints ---
found = {}
for p in (sorted(INPUT.rglob("*.pt")) if INPUT.exists() else []):
    m, tgt = _RID.search(p.as_posix()), _target(p.name)
    if m and tgt:
        found.setdefault((m.group(0), tgt), p)     # first match per (run_id, kind)

if not found:
    print("No orientation/scale_strong_seed*.pt under /kaggle/input -- fresh start "
          "(Add Input -> a prior version's output to resume).")
for (rid, tgt), src in sorted(found.items()):
    dst = ENC / rid / tgt
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dst)
    print(f"restored {rid}/{tgt:12s} <- {src.name}")

# --- probe artifacts (stacks.npz, meta.json, per-encoder _cache/*.npz) ---
n_probe = 0
for p in (sorted(INPUT.rglob("stacks.npz")) + sorted(INPUT.rglob("meta.json"))
          + sorted(INPUT.rglob("_cache/*.npz")) if INPUT.exists() else []):
    m = _PROBE.search(p.as_posix())
    if not m:
        continue
    dst = PROB / m.group(1)
    if dst.exists():
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(p, dst)
    n_probe += 1
    print(f"restored probes/{m.group(1)}")
print(f"restored {n_probe} probe artifact(s)")

## 6. Integrity check — keep good checkpoints, purge corrupt ones
Each restored `backbone.pt` / `last_ckpt.pt` is opened as a zip archive. A valid
`backbone.pt` means the seed is done (section 7 skips it). A corrupt `backbone.pt`
is deleted with its `last_ckpt.pt` so the seed retrains from scratch; a valid
`last_ckpt.pt` with no backbone lets section 7 resume mid-training. This guards
against a truncated upload crashing the resume load. Runs over both conditions.

In [ ]:
import zipfile
from pathlib import Path

ENC = Path("/kaggle/working/probe-capacity-invariance/results/encoders")

def _state(p):
    if not p.exists():
        return "missing"
    try:
        return None if zipfile.ZipFile(p).testzip() is None else "corrupt"
    except Exception as e:
        return f"not-a-zip ({e})"

dirs = sorted(list(ENC.glob("orientation_strong_seed*")) + list(ENC.glob("scale_strong_seed*")))
done, resume, fresh = [], [], []
for d in dirs:
    bb, ck = d / "backbone.pt", d / "last_ckpt.pt"
    bstat = _state(bb)
    if bstat is None:
        done.append(d.name); print(f"{d.name:30s} backbone OK -> skip"); continue
    if bstat != "missing":                          # corrupt backbone -> drop both
        bb.unlink(missing_ok=True); ck.unlink(missing_ok=True)
        fresh.append(d.name); print(f"{d.name:30s} backbone {bstat} -> PURGED (retrain)"); continue
    cstat = _state(ck)                              # no backbone: keep a good last_ckpt
    if cstat is None:
        resume.append(d.name); print(f"{d.name:30s} resume from last_ckpt.pt")
    else:
        ck.unlink(missing_ok=True)
        fresh.append(d.name); print(f"{d.name:30s} last_ckpt {cstat} -> from scratch")
print(f"\ndone={len(done)}  resume={len(resume)}  fresh={len(fresh)}")

## 7. Train — orientation_strong + scale_strong, seeds 0–11 (resume-aware)
One unified 24-job queue (12 orientation + 12 scale), one `train_simclr` per GPU
pinned via `CUDA_VISIBLE_DEVICES`, each training from random init to 50 epochs.
The queue is **interleaved** (orientation/scale/orientation/…) so both conditions
progress together — on 2 GPUs the first two jobs are orientation_seed0 and
scale_seed0 side by side. Per-seed logs in `results/sweep_logs/`.

Idempotent + resumable — **re-run this cell to continue** after a session ends;
done seeds skip, partial seeds resume from their own `last_ckpt.pt`.

Check section 9's diagnostics after the first seeds of each condition land (new
conditions — verify the cut before spending the rest of the quota).

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
ENC = REPO / "results" / "encoders"
LOGS = REPO / "results" / "sweep_logs"; LOGS.mkdir(parents=True, exist_ok=True)

# Two Shapes3D cells sharing one image cache and one GPU queue.
CONFIGS = {
    "orientation_strong": "configs/run/orientation_strong.yaml",
    "scale_strong":       "configs/run/scale_strong.yaml",
}
SEEDS = list(range(12))            # 12 seeds/cell (H2 decidability; buffers gate failures)
N_GPUS = max(1, torch.cuda.device_count())

# Interleave the two conditions so both advance together ("train both at once").
# For condition-major instead (finish one full cell first, so you can probe+ship
# it while the other trains), use: [(t, s) for t in CONFIGS for s in SEEDS].
JOBS = [(tag, s) for s in SEEDS for tag in CONFIGS]

def run_id(tag, seed): return f"{tag}_seed{seed}"
def is_done(tag, seed): return (ENC / run_id(tag, seed) / "backbone.pt").exists()

def _launch(tag, seed, gpu):
    rid = run_id(tag, seed)
    env = dict(os.environ,
               CUDA_VISIBLE_DEVICES=str(gpu),
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    log = open(LOGS / f"{rid}.log", "a")
    cmd = [sys.executable, "-m", "src.encoders.train_simclr", "--config", CONFIGS[tag],
           "--set", f"run.seed={seed}", "run.num_workers=2", "run.device=cuda"]
    p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT)
    return {"proc": p, "log": log, "rid": rid}

def run(n_gpus=N_GPUS, poll=15):
    subprocess.run(["pkill", "-9", "-f", "src.encoders.train_simclr"])  # reap orphans
    time.sleep(2)
    queue = [(t, s) for (t, s) in JOBS if not is_done(t, s)]
    print(f"{len(JOBS)} jobs | {len(JOBS)-len(queue)} done | {len(queue)} to run on {n_gpus} GPU(s)")
    slots = [None] * n_gpus
    try:
        while queue or any(slots):
            for g in range(n_gpus):
                if slots[g] is None and queue:
                    tag, seed = queue.pop(0)
                    if is_done(tag, seed):
                        continue
                    slots[g] = _launch(tag, seed, g)
                    print(f"[GPU{g}] start  {slots[g]['rid']}", flush=True)
            time.sleep(poll)
            for g in range(n_gpus):
                s = slots[g]
                if s and s["proc"].poll() is not None:
                    s["log"].close()
                    ok = (ENC / s["rid"] / "backbone.pt").exists()
                    print(f"[GPU{g}] {'DONE ' if ok else 'EXIT(rc=%s) ' % s['proc'].returncode}{s['rid']}", flush=True)
                    slots[g] = None
    except KeyboardInterrupt:
        for s in slots:
            if s: s["proc"].terminate()
        print("interrupted — checkpoints are saved; re-run this cell to resume")
        return
    print("done — all 24 encoders have a backbone.pt")

run()

## 8. Monitor
Re-run any time while section 7 (or 10) is running. Tail a specific job's log
with e.g. `!tail -n 30 results/sweep_logs/orientation_strong_seed0.log`.

In [ ]:
!nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv

## 9. Progress + final diagnostics
Per-seed state per condition and, once done, the collapse diagnostics for each encoder.

In [ ]:
import json as _json

for tag in ("orientation_strong", "scale_strong"):
    print(f"== {tag} ==")
    for seed in range(12):
        rid = f"{tag}_seed{seed}"; d = ENC / rid
        if (d / "backbone.pt").exists():
            m = _json.loads((d / "metrics.json").read_text()); diag = m.get("diagnostics", {})
            print(f"  {rid}: DONE loss={m['final_loss']:.4f} nan={m['nan_aborted']} epochs={m['epochs_run']} | "
                  f"feat_std={diag.get('feat_std'):.4f} eff_rank={diag.get('eff_rank'):.1f} "
                  f"align={diag.get('alignment'):.3f} unif={diag.get('uniformity'):.3f}")
        elif (d / "last_ckpt.pt").exists():
            last = None
            for line in (d / "train_log.jsonl").read_text().splitlines():
                try: last = _json.loads(line).get("epoch", last)
                except Exception: pass
            print(f"  {rid}: partial @ep{last}")
        else:
            print(f"  {rid}: todo")

## 10. Probe both cells → `stacks.npz` (the interface contract)
Fits the probe-capacity ladder over each cell's 12 encoders + 12 seed-paired
random-encoder floor seeds and writes `results/probes/<condition>_strong/{stacks.npz,
meta.json}` — the only thing the H1–H4 statistics session depends on. Both cells
are probed **in parallel** (one per GPU); each is **resumable per-encoder**
(`--resume` caches every finished encoder under `_cache/`, restored by section 5),
and a cell whose `stacks.npz` already exists is skipped. The per-encoder quality
gate runs inside the step. Watch progress with
`!tail -n 30 results/sweep_logs/probe_orientation_strong.log`.

Expect **~3–4 h** total on 2× T4 (both cells at once).

In [ ]:
import os, sys, subprocess, time, glob
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
LOGS = REPO / "results" / "sweep_logs"; LOGS.mkdir(parents=True, exist_ok=True)
CELLS = {                         # run_id tag -> augmentation condition
    "orientation_strong": "orientation",
    "scale_strong":       "scale",
}
N_SEEDS = 12
RANDOM_SEEDS = [str(i) for i in range(N_SEEDS)]  # seed-paired to trained for epsilon_G
N_GPUS = max(1, torch.cuda.device_count())

def _launch_probe(tag, condition, gpu):
    encs = sorted(glob.glob(str(REPO / f"results/encoders/{tag}_seed*/backbone.pt")))
    assert len(encs) == N_SEEDS, f"{tag}: only {len(encs)}/{N_SEEDS} encoders trained -- finish section 7 first"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu),
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    log = open(LOGS / f"probe_{tag}.log", "a")
    cmd = [sys.executable, "-m", "src.probes.run_sweep",
           "--config", "configs/probe/ladder.yaml",
           "--dataset", "shapes3d",
           "--condition", condition, "--strength", "strong",
           "--encoders", *encs,
           "--random-seed", *RANDOM_SEEDS,
           "--epochs", "100", "--num-workers", "2", "--resume"]
    p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT)
    return {"proc": p, "log": log, "tag": tag}

todo = []
for tag, cond in CELLS.items():
    if (REPO / "results" / "probes" / tag / "stacks.npz").exists():
        print(f"{tag}: stacks.npz already present -> skip.")
    else:
        todo.append((tag, cond))

t0 = time.time()
for base in range(0, len(todo), N_GPUS):          # waves of N_GPUS (parallel on 2× T4)
    wave = todo[base:base + N_GPUS]
    procs = [_launch_probe(tag, cond, g) for g, (tag, cond) in enumerate(wave)]
    for pr in procs:
        print(f"[GPU{procs.index(pr)}] probing {pr['tag']} (resumable)", flush=True)
    for pr in procs:
        rc = pr["proc"].wait(); pr["log"].close()
        print(f"{pr['tag']}: {'OK' if rc == 0 else 'FAILED (exit %s)' % rc}", flush=True)
print(f"probe step done in {(time.time()-t0)/3600:.2f} h")

## 11. Save the finished run
**Save Version (Commit)** to persist `/kaggle/working`. Deliverables:
- 24 `backbone.pt` under `results/encoders/{orientation,scale}_strong_seed{0..11}/`;
- `results/probes/{orientation_strong,scale_strong}/{stacks.npz, meta.json}`.

With these, both the orientation and scale arms are ready for the statistics layer.
Between sessions, `Add Input -> this version's output` so section 5 restores every
encoder checkpoint and probe cache before you continue.

## Time estimate (rough)
Basis: ~5 GPU-h per full Shapes3D encoder to the 50-epoch cut on a T4 (the
measured per-encoder cost from the earlier single-condition cells).

- **Training:** 24 encoders × ~5 GPU-h = ~120 GPU-h. On **2× T4** that is
  ~120 ÷ 2 = **~60 h wall-clock**. At Kaggle's ~30 GPU-h/week quota and ~12 h
  session cap: **≈ 6 sessions ≈ ~2 weeks**. On a single T4 it is ~120 h (double).
- **Probe:** ~3–4 h per cell; both cells run in parallel on 2× T4 -> **~3–4 h**
  total (~6–8 h if serial on one GPU).
- **Total: ~63–64 h wall-clock on 2× T4** (~2 weeks of quota), plus whatever
  Save-Version/restore overhead each session adds.

These are order-of-magnitude; the real per-encoder time depends on T4 throughput
and dataloader speed. After the first two encoders finish, divide their observed
wall-time by 2 (two GPUs) for a calibrated projection.